# 15 — Eventi su clip NBA (v2): palla ad alta risoluzione, tagli di scena, lavagna automatica e semi-automatica

Seconda versione del modulo eventi, pensata per **clip NBA registrate** (dominio del detector) e costruita sugli errori visti nel notebook 14:

| Problema visto nel 14 | Soluzione nel 15 |
|---|---|
| Palla trovata solo nel 37% dei frame, scambiata con altro | Detection a **risoluzione piena** (`DET_IMGSZ = 1280`) + **tracker della palla** (filtro di Kalman con finestra di ricerca): si accetta solo una palla coerente con il moto precedente |
| La palla "schizza" | Niente interpolazione lunga: si riempiono solo buchi ≤ 4 frame tra punti accettati |
| Le riprese NBA hanno **tagli di inquadratura** e replay | Rilevamento automatico dei tagli: tracker, possesso ed eventi ripartono da zero a ogni taglio; si possono escludere intervalli (replay) |
| Video lunghi non entrano in memoria | I frame vengono letti dal video a ogni passaggio, non tenuti tutti in RAM |
| Eventi automatici inaffidabili | Oltre agli eventi automatici, **modalità semi-automatica**: gli eventi li scrivi tu in un CSV (tempo, tipo, ID dei giocatori visti nel video), posizioni e squadre restano automatiche. Lo stesso CSV fa da **riferimento** per valutare gli eventi automatici |

Output in `events_nba/<clip>/`, compreso un **video diagnostico** con tutte le palle candidate (grigio) e quella accettata (giallo).
Runtime: **Colab GPU T4**. Servono `models/yolo11m_ft.pt` (salvato dal notebook 10) e il Secret `ROBOFLOW_API_KEY`.

In [ ]:
%pip install -q ultralytics trackers supervision inference "git+https://github.com/roboflow/sports.git@feat/basketball"
# dopo il pip: Runtime -> Riavvia sessione, poi esegui dalla cella successiva

In [ ]:
import sys
if 'triton' not in sys.modules: sys.modules['triton'] = None     # evita il crash di triton su Colab

# ==== SETUP COMUNE (uguale in tutti i notebook) ====
import sys, os
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Tesi_Basket')   # progetto su Drive
    FAST = Path('/content/fast')                                 # disco locale Colab (veloce)
else:
    ROOT = Path(r'F:\Tesi Basket')                               # progetto in locale
    FAST = ROOT / '_cache'
FAST.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Cartella del progetto non trovata: {ROOT}'
print('Progetto:', ROOT)

import faulthandler; faulthandler.enable()
import json, time, subprocess
import numpy as np, pandas as pd, cv2
import supervision as sv
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

## 1. Configurazione

Metti la clip in `videos/` su Drive. Consigli per la registrazione: partita intera o *condensed game* (non highlights), inquadratura larga, 1280×720 o più, 30 fps, senza mouse o sovrimpressioni del player. Se la clip contiene replay, indicali in `ESCLUDI`.

In [ ]:
VIDEO = ROOT / 'videos' / 'nba_clip_01.mp4'     # <-- la tua clip NBA
START_S, END_S = 0, None                        # tratto da analizzare (secondi); None = fino alla fine
ESCLUDI = []                                    # intervalli da ignorare, es. replay: [(42.0, 51.5)]
LEAGUE_NAME = 'nba'                             # 'nba' oppure 'fiba'

WORK_W = 1280                                   # larghezza di lavoro (i video più grandi vengono ridotti)
DET_WEIGHTS = ROOT / 'models' / 'yolo11m_ft.pt'      # su clip NBA (in-domain) YOLO11m è il migliore in E1 ed è 3,5x più veloce
DET_IMGSZ = 1280                                # risoluzione del detector: alta = palla più visibile (più lento)
BALL_WEIGHTS = ROOT / 'models' / 'ball_detector.pt'   # detector dedicato alla palla (notebook 16); se manca si usa la palla del detector principale
BALL_IMGSZ = 1280
DET_CONF = 0.05
COURT_MODEL_ID = 'basketball-court-detection-2/14'
HOMO_EVERY = 3
CUT_THR = 0.55                                  # somiglianza minima tra frame consecutivi; sotto = taglio di scena
FT_TO_M = 0.3048

# tracce e squadre (come notebook 13/14)
INTERP_MAX_GAP = 20; DUP_IOU = 0.7; SPLIT_MIN_SAMPLES = 5; OUTLIER_K = 3.0
SCOREBOARD_REL = (0.00, 0.79, 0.82, 0.95)       # tabellone NBA in basso; None se non c'è

# palla (tracker di Kalman)
BALL_MIN_CONF = 0.15        # candidati palla considerati
BALL_INIT_CONF = 0.40       # confidenza per (ri)agganciare la palla
BALL_GATE = 0.05            # raggio di ricerca per frame di assenza (frazione della larghezza)
BALL_MAX_MISS = 8           # frame senza conferma dopo i quali la palla è persa
BALL_FILL = 4               # buchi riempiti tra punti accettati

# possesso ed eventi (come notebook 14 v2)
POSS_CONF = 0.30; POSS_RADIUS = 0.35; POSS_MIN = 5; POSS_LOST = 10
MIN_SEG_S = 0.6; STEAL_MIN_S = 1.5; PASS_MIN_M = 1.0; SWITCH_DIST_M = 1.0; PASS_MAX_GAP_S = 2.0
SHOT_MIN_FRAMES = 3; SHOT_WINDOW_S = 3.0; MADE_CONF = 0.40; MADE_MIN_FRAMES = 2
EVENT_TOL_S = 1.0

MANUAL = ROOT / 'annotations' / 'eventi_manuali.csv'   # eventi scritti a mano (lavagna semi-automatica + riferimento)
CLIP_ID = VIDEO.stem
OUT = ROOT / 'events_nba' / CLIP_ID; OUT.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO)); assert cap.isOpened(), f'Video non trovato: {VIDEO}'
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0; TOT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
SRC_W, SRC_H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); cap.release()
F0 = int(START_S * FPS); F1 = min(TOT, int(END_S * FPS)) if END_S else TOT
N = F1 - F0
W_IMG = min(WORK_W, SRC_W); H_IMG = int(round(SRC_H * W_IMG / SRC_W))
print(f'{CLIP_ID}: {SRC_W}x{SRC_H} a {FPS:.2f} fps -> analizzo {N} frame ({N / FPS:.1f} s) a {W_IMG}x{H_IMG}')
excluded = np.zeros(N, bool)
for a, b in ESCLUDI: excluded[max(0, int((a - START_S) * FPS)):max(0, int((b - START_S) * FPS))] = True

def frames_iter(step=1, wanted=None, desc=None):
    # legge i frame dal video uno alla volta (niente video intero in memoria)
    cap = cv2.VideoCapture(str(VIDEO))
    for _ in range(F0): cap.grab()
    for fi in tqdm(range(N), desc=desc, leave=False, disable=desc is None):
        ok, f = cap.read()
        if not ok: break
        if fi % step or (wanted is not None and fi not in wanted): continue
        yield fi, (cv2.resize(f, (W_IMG, H_IMG), interpolation=cv2.INTER_AREA) if f.shape[1] != W_IMG else f)
    cap.release()

## 2. Detection ad alta risoluzione, tagli di scena e tracking (BoT-SORT, riavviato a ogni taglio)

In [ ]:
from ultralytics import YOLO, RTDETR
import trackers as T
model = RTDETR(str(DET_WEIGHTS)) if 'rtdetr' in str(DET_WEIGHTS).lower() else YOLO(str(DET_WEIGHTS))
NAMES = model.names

def iou_xyxy(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-9)

USE_BALL = BALL_WEIGHTS.is_file()
ball_model = YOLO(str(BALL_WEIGHTS)) if USE_BALL else None
BALL_ID = [i for i, n in NAMES.items() if n == 'ball'][0]
print('palla:', 'detector dedicato (notebook 16)' if USE_BALL else 'detector principale')
tag = f's{F0}_n{N}_w{W_IMG}_i{DET_IMGSZ}' + ('_ball' if USE_BALL else '')
dcache, ccache = OUT / f'detections_{tag}.npy', OUT / f'cuts_{tag}.json'
have_det = dcache.is_file() and ccache.is_file()
D = np.load(dcache) if have_det else None
det_by = {}
if have_det and len(D):
    D = D[np.argsort(D[:, 0], kind='stable')]; u, idx = np.unique(D[:, 0].astype(int), return_index=True)
    det_by = {int(k): a for k, a in zip(u, np.split(D, idx[1:]))}
PLAYER_IDS = [i for i, n in NAMES.items() if n.startswith('player')]
rows_det, cuts, prev_h = [], [] if not have_det else json.loads(ccache.read_text()), None
rows_trk, tracker, scene = [], None, -1
cut_set = set(cuts)
for fi, f in frames_iter(desc='detection + tracking'):
    if not have_det:
        small = cv2.cvtColor(cv2.resize(f, (160, 90)), cv2.COLOR_BGR2HSV)
        h = cv2.calcHist([small], [0, 1], None, [30, 16], [0, 180, 0, 256]); cv2.normalize(h, h)
        if prev_h is not None and cv2.compareHist(prev_h, h, cv2.HISTCMP_CORREL) < CUT_THR: cuts.append(fi); cut_set.add(fi)
        prev_h = h
        r = model.predict(f, imgsz=DET_IMGSZ, conf=DET_CONF, iou=0.7, verbose=False)[0]
        d = np.column_stack([np.full(len(r.boxes), fi), r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()])
        if USE_BALL:                                          # la palla viene dal detector dedicato
            rb = ball_model.predict(f, imgsz=BALL_IMGSZ, conf=0.05, verbose=False, max_det=10)[0]
            db = np.column_stack([np.full(len(rb.boxes), fi), rb.boxes.xyxy.cpu().numpy(), rb.boxes.conf.cpu().numpy(), np.full(len(rb.boxes), BALL_ID)])
            d = np.concatenate([d[d[:, 6].astype(int) != BALL_ID], db])
        rows_det.append(d); det_by[fi] = d
    if tracker is None or fi in cut_set:
        tracker = T.BoTSORTTracker(frame_rate=FPS); scene += 1
    d = det_by.get(fi, np.zeros((0, 7)))
    p = d[np.isin(d[:, 6].astype(int), PLAYER_IDS)] if len(d) else d
    tr = tracker.update(sv.Detections(xyxy=p[:, 1:5].astype(np.float32), confidence=p[:, 5].astype(np.float32),
                                      class_id=np.zeros(len(p), int)), frame=f)
    if len(tr) == 0: continue
    M = iou_xyxy(tr.xyxy, p[:, 1:5]) if len(p) else np.zeros((len(tr), 0))
    for k, (b, tid) in enumerate(zip(tr.xyxy, tr.tracker_id)):
        if tid is None or tid < 0: continue
        j = int(M[k].argmax()) if M.shape[1] else -1
        ok = j >= 0 and M[k, j] > 0.5
        rows_trk.append([fi, scene * 1000 + int(tid) + 1, *b, NAMES[int(p[j, 6])] if ok else 'player', float(p[j, 5]) if ok else np.nan])
if not have_det:
    D = np.concatenate(rows_det) if rows_det else np.zeros((0, 7)); np.save(dcache, D); ccache.write_text(json.dumps(cuts))
det = pd.DataFrame(D, columns=['frame', 'x1', 'y1', 'x2', 'y2', 'conf', 'cls'])
det['frame'] = det.frame.astype(int); det['name'] = det.cls.astype(int).map(NAMES).astype(str)
scene_of = np.zeros(N, int)
for c in cuts: scene_of[c:] += 1
tracks = pd.DataFrame(rows_trk, columns=['frame', 'track_id', 'x1', 'y1', 'x2', 'y2', 'cls', 'cls_conf'])
tracks = tracks[~excluded[tracks.frame.values]].reset_index(drop=True)

def interpolate(t, max_gap, cols=('x1', 'y1', 'x2', 'y2')):
    out = []
    for tid, g in t.groupby('track_id'):
        g = g.set_index('frame').reindex(range(int(g.frame.min()), int(g.frame.max()) + 1))
        miss = g.x1.isna(); run = (miss != miss.shift()).cumsum()
        g[list(cols)] = g[list(cols)].interpolate(limit_area='inside')
        keep = ~miss | (miss.groupby(run).transform('sum') <= max_gap)
        g = g[keep].copy(); g['track_id'] = tid; g['cls'] = g['cls'].fillna('interp')
        g.index.name = 'frame'; out.append(g.reset_index())
    return pd.concat(out, ignore_index=True)
tracks = interpolate(tracks, INTERP_MAX_GAP)
tracks['frame'] = tracks.frame.astype(int); tracks['track_id'] = tracks.track_id.astype(int)
print(f'tagli di scena: {len(cuts)} ({[round(c / FPS + START_S, 1) for c in cuts][:15]} s) | tracce: {tracks.track_id.nunique()}')
print(det[det.conf >= 0.3]['name'].value_counts().to_dict())

## 3. Pulizia tracce, squadre e arbitro (come notebook 13/14)

In [ ]:
length = tracks.groupby('track_id').size(); drop = []
for fi, g in tracks.groupby('frame'):
    if len(g) < 2: continue
    M = iou_xyxy(g[['x1', 'y1', 'x2', 'y2']].values, g[['x1', 'y1', 'x2', 'y2']].values); np.fill_diagonal(M, 0)
    ids, idx = g.track_id.values, g.index.values
    for i, j in zip(*np.where(np.triu(M) > DUP_IOU)):
        drop.append(idx[i] if length[ids[i]] < length[ids[j]] else idx[j])
tracks = tracks.drop(index=sorted(set(drop))).reset_index(drop=True)
tracks['foot_x'] = (tracks.x1 + tracks.x2) / 2; tracks['foot_y'] = tracks.y2
tracks['foot_ok'] = tracks.y2 < H_IMG - 3
if SCOREBOARD_REL:
    sx1, sy1, sx2, sy2 = SCOREBOARD_REL[0] * W_IMG, SCOREBOARD_REL[1] * H_IMG, SCOREBOARD_REL[2] * W_IMG, SCOREBOARD_REL[3] * H_IMG
    tracks.loc[tracks.foot_x.between(sx1, sx2) & tracks.y2.between(sy1 - 6, sy2), 'foot_ok'] = False

# feature della maglia: un solo passaggio sul video, solo sui frame necessari
need = set(range(0, N, 3)); court_px = []
feat_rows = []
by_f = dict(tuple(tracks[tracks.frame % 3 == 0].groupby('frame')))
for fi, f in frames_iter(step=3, desc='colori maglia'):
    if fi % 30 == 0:
        court_px.append(cv2.cvtColor(np.ascontiguousarray(f[int(.6*H_IMG):int(.78*H_IMG), int(.2*W_IMG):int(.8*W_IMG)]), cv2.COLOR_BGR2LAB).reshape(-1, 3))
    g = by_f.get(fi)
    if g is None: continue
    for r in g.itertuples():
        x1, y1, x2, y2 = max(0, int(r.x1)), max(0, int(r.y1)), min(W_IMG, int(r.x2)), min(H_IMG, int(r.y2))
        w, h = x2 - x1, y2 - y1
        if h < 30 or w < 8: continue
        crop = np.ascontiguousarray(f[y1 + int(.15*h):y1 + int(.5*h), x1 + int(.2*w):x2 - int(.2*w)])
        if crop.size: feat_rows.append((r.track_id, fi, cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)))
COURT_LAB = np.median(np.concatenate(court_px).astype(np.float32), axis=0)
def hist_of(lab):
    lab = lab[np.linalg.norm(lab - COURT_LAB, axis=1) > 25]
    if len(lab) < 20: return None
    h = np.histogramdd(lab, bins=(4, 6, 6), range=((0, 256), (0, 256), (0, 256)))[0].ravel(); return h / h.sum()
F, K, R = [], [], []
for tid, fi, lab in feat_rows:
    h = hist_of(lab)
    if h is not None: F.append(h); K.append(tid); R.append(fi)
F, K, R = np.array(F), np.array(K), np.array(R)

class KMeans2:
    def __init__(self, n_init=20, iters=100, seed=0): self.n_init, self.iters, self.rng = n_init, iters, np.random.default_rng(seed)
    def fit(self, X):
        X = np.asarray(X, np.float64); best = None
        for _ in range(self.n_init):
            C = X[self.rng.choice(len(X), 2, replace=len(X) < 2)].copy()
            for _ in range(self.iters):
                lab = self._assign(X, C)
                newC = np.array([X[lab == k].mean(0) if (lab == k).any() else C[k] for k in range(2)])
                if np.allclose(newC, C): break
                C = newC
            inertia = ((X - C[self._assign(X, C)]) ** 2).sum()
            if best is None or inertia < best[0]: best = (inertia, C)
        self.cluster_centers_ = best[1]; return self
    @staticmethod
    def _assign(X, C): return np.argmin(((X[:, None, :] - C[None]) ** 2).sum(-1), axis=1)
    def predict(self, X): return self._assign(np.asarray(X, np.float64), self.cluster_centers_)

def fit_teams(F, K):
    med = {int(t): np.median(F[K == t], axis=0) for t in np.unique(K)}
    n = pd.Series(K).value_counts()
    fit_ids = [t for t in med if n[t] >= 5] or list(med)
    outliers = set()
    for _ in range(2):
        km = KMeans2().fit(np.array([med[t] for t in fit_ids if t not in outliers]))
        lum = [(c.reshape(4, 6, 6).sum((1, 2)) * np.array([32, 96, 160, 224])).sum() for c in km.cluster_centers_]
        if lum[0] > lum[1]: km.cluster_centers_ = km.cluster_centers_[::-1].copy()
        team = {t: int(km.predict(med[t][None])[0]) for t in med}
        d = pd.Series({t: np.linalg.norm(med[t] - km.cluster_centers_[team[t]]) for t in med})
        dl = d[[t for t in fit_ids if t not in outliers]]
        thr = dl.median() + OUTLIER_K * max((dl - dl.median()).abs().median() * 1.4826, 0.25 * dl.median(), 1e-6)
        outliers |= set(d.index[d > thr])
    for t in outliers: team[t] = -1
    return km, team

km, team_of = fit_teams(F, K)
n_split = 0; lab_s = km.predict(F); next_id = int(tracks.track_id.max()) + 1; remap = []
for tid in np.unique(K):
    if team_of[int(tid)] < 0: continue
    sel = np.argsort(R[K == tid]); f_t = R[K == tid][sel]; l_t = lab_s[K == tid][sel]
    other = l_t != team_of[int(tid)]; run_start, cnt, cuts_t = None, 0, []
    for k, o in enumerate(other):
        if o: run_start = k if cnt == 0 else run_start; cnt += 1
        else:
            if cnt >= SPLIT_MIN_SAMPLES: cuts_t.append((f_t[run_start], f_t[k]))
            cnt = 0
    if cnt >= SPLIT_MIN_SAMPLES: cuts_t.append((f_t[run_start], N))
    for a, b in cuts_t:
        m = (tracks.track_id == tid) & (tracks.frame >= a) & (tracks.frame < b)
        tracks.loc[m, 'track_id'] = next_id; remap.append((tid, a, b, next_id)); next_id += 1; n_split += 1
for tid, a, b, nid in remap:
    m = (K == tid) & (R >= a) & (R < b); K[m] = nid
if n_split: km, team_of = fit_teams(F, K)
ref_by = {f: g[['x1', 'y1', 'x2', 'y2']].values for f, g in det[(det['name'] == 'referee') & (det.conf >= 0.3)].groupby('frame')}
for tid, g in tracks[tracks.frame % 3 == 0].groupby('track_id'):
    hits = [f in ref_by and iou_xyxy(np.array([[x1, y1, x2, y2]]), ref_by[f]).max() > 0.5
            for f, x1, y1, x2, y2 in g[['frame', 'x1', 'y1', 'x2', 'y2']].itertuples(index=False)]
    if hits and np.mean(hits) >= 0.3: team_of[int(tid)] = -1
tracks['team'] = tracks.track_id.map(team_of).fillna(-1).astype(int)
print('tracce divise:', n_split, '| tracce per squadra:', pd.Series(team_of).value_counts().to_dict(),
      '| escluse:', len([t for t, k in team_of.items() if k < 0]))
TEAM_COLORS = {0: (255, 140, 30), 1: (40, 40, 230), -1: (160, 160, 160)}    # BGR: squadra 0 (scura) blu, 1 (chiara) rossa

# foglio di controllo: un ritaglio per traccia lunga
long_ids = tracks.groupby('track_id').size().loc[lambda s: s >= FPS].index
mid = {tid: int(g.frame.iloc[len(g) // 2]) for tid, g in tracks[tracks.track_id.isin(long_ids)].groupby('track_id')}
crops = {}
want = {f: [t for t, ff in mid.items() if ff == f] for f in set(mid.values())}
for fi, f in frames_iter(wanted=set(want), desc='foglio squadre'):
    for tid in want[fi]:
        r = tracks[(tracks.track_id == tid) & (tracks.frame == fi)].iloc[0]
        c = np.ascontiguousarray(f[max(0, int(r.y1)):min(H_IMG, int(r.y2)), max(0, int(r.x1)):min(W_IMG, int(r.x2))])
        if c.size and c.shape[0] > 1 and c.shape[1] > 1:
            c = cv2.resize(c, (60, 120)); cv2.putText(c, str(tid), (2, 14), cv2.FONT_HERSHEY_SIMPLEX, .45, (0, 255, 255), 2); crops[tid] = c
fig, axes = plt.subplots(3, 1, figsize=(16, 7.5))
for ax, team in zip(axes, [0, 1, -1]):
    cc = [crops[t] for t in sorted(crops) if team_of.get(t, -1) == team][:25]
    if cc: ax.imshow(cv2.cvtColor(np.hstack(cc), cv2.COLOR_BGR2RGB))
    ax.set_title(f'Squadra {team}' if team >= 0 else 'Escluse (arbitro / anomale)'); ax.axis('off')
fig.tight_layout(); fig.savefig(OUT / 'controllo_squadre.png', dpi=130); plt.show()

## 4. Omografia (per scena) e posizioni sul campo

In [ ]:
from sports import MeasurementUnit
from sports.basketball import CourtConfiguration, League, draw_court
LEAGUE = League.FIBA if LEAGUE_NAME == 'fiba' else League.NBA
court = CourtConfiguration(league=LEAGUE, measurement_unit=MeasurementUnit.FEET)
V = np.asarray(court.vertices, dtype=np.float32)
L_FT, W_FT = float(court.court_length), float(court.court_width)

hfile = OUT / f'homographies_{tag}_{LEAGUE.value}.json'
if hfile.is_file():
    hs = {int(k): np.asarray(v) for k, v in json.loads(hfile.read_text()).items()}
else:
    from inference import get_model
    try:
        from google.colab import userdata; API_KEY = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        API_KEY = os.environ.get('ROBOFLOW_API_KEY') or __import__('getpass').getpass('API key Roboflow: ')
    court_model = get_model(model_id=COURT_MODEL_ID, api_key=API_KEY)
    hs = {}
    for fi, f in frames_iter(step=HOMO_EVERY, desc='omografia'):
        if excluded[fi]: continue
        res = court_model.infer(f, confidence=0.3)[0]
        if not res.predictions: continue
        pr = max(res.predictions, key=lambda p: p.confidence)
        pts = [p for p in pr.keypoints if p.confidence >= 0.5 and 0 <= p.class_id < len(V)]
        if len(pts) >= 6:
            fwd, mask = cv2.findHomography(V[[p.class_id for p in pts]], np.float32([[p.x, p.y] for p in pts]), cv2.RANSAC, 8.0)
            if fwd is not None and mask is not None and mask.sum() >= 6:
                try: hs[fi] = np.linalg.inv(fwd)
                except np.linalg.LinAlgError: pass
    hfile.write_text(json.dumps({str(k): v.tolist() for k, v in hs.items()}))
# omografia del frame calcolato più vicino, ma solo nella stessa scena (mai attraverso un taglio)
Hs = [None] * N
for sc in np.unique(scene_of):
    fr_sc = np.where(scene_of == sc)[0]; ks = np.array([k for k in hs if scene_of[k] == sc])
    if len(ks) == 0: continue
    for fi in fr_sc: Hs[fi] = hs[int(ks[np.abs(ks - fi).argmin()])]
print(f'omografie valide: {len(hs)} | frame senza omografia: {sum(h is None for h in Hs)} su {N}')

def to_court(x, y, fi):
    H = Hs[int(fi)]
    if H is None: return (np.nan, np.nan)
    p = cv2.perspectiveTransform(np.float32([[[x, y]]]), H)[0, 0]; return float(p[0]), float(p[1])
xy = [to_court(r.foot_x, r.foot_y, r.frame) if r.foot_ok else (np.nan, np.nan) for r in tracks.itertuples()]
tracks[['x_ft', 'y_ft']] = np.array(xy)
tracks.loc[~(tracks.x_ft.between(-3, L_FT + 3) & tracks.y_ft.between(-3, W_FT + 3)), ['x_ft', 'y_ft']] = np.nan
W_SM = max(3, int(round(FPS / 2)) | 1)
tracks = tracks.sort_values(['track_id', 'frame']).reset_index(drop=True)
tracks[['xs_ft', 'ys_ft']] = tracks.groupby('track_id')[['x_ft', 'y_ft']].transform(
    lambda s: s.rolling(W_SM, center=True, min_periods=1).median().rolling(5, center=True, min_periods=1).mean())
tracks.to_csv(OUT / 'tracks_court.csv', index=False)
print('posizioni sul campo valide:', int(tracks.xs_ft.notna().sum()), 'su', len(tracks))

## 5. Palla: tracker di Kalman

Per ogni frame si considerano tutti i candidati `ball` (≥ `BALL_MIN_CONF`). Se la palla è agganciata, si accetta solo il candidato più vicino alla posizione **prevista** dal moto (entro un raggio che cresce con i frame di assenza); se nessun candidato è plausibile la palla resta "prevista" per al massimo `BALL_MAX_MISS` frame, poi è persa. Si riaggancia solo con una detection sicura (≥ `BALL_INIT_CONF`). Il tracker riparte a ogni taglio di scena.

In [ ]:
def new_kf(x, y):
    kf = cv2.KalmanFilter(4, 2)
    kf.transitionMatrix = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], np.float32)
    kf.measurementMatrix = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], np.float32)
    kf.processNoiseCov = np.diag([1, 1, 8, 8]).astype(np.float32)
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 4
    kf.statePost = np.array([[x], [y], [0], [0]], np.float32); kf.errorCovPost = np.eye(4, dtype=np.float32) * 50
    return kf

bc = det[(det['name'] == 'ball') & (det.conf >= BALL_MIN_CONF)].copy()
bc['cx'] = (bc.x1 + bc.x2) / 2; bc['cy'] = (bc.y1 + bc.y2) / 2
cand_by = {f: g[['cx', 'cy', 'conf']].values for f, g in bc.groupby('frame')}
acc = np.full((N, 2), np.nan); kf, miss, reinit = None, 0, 0
for fi in range(N):
    if excluded[fi] or (fi > 0 and scene_of[fi] != scene_of[fi - 1]): kf, miss = None, 0
    if excluded[fi]: continue
    C = cand_by.get(fi, np.zeros((0, 3)))
    if kf is not None:
        pred = kf.predict()[:2].ravel()
        if len(C):
            d = np.hypot(C[:, 0] - pred[0], C[:, 1] - pred[1]); j = int(d.argmin())
            if d[j] <= BALL_GATE * W_IMG * (1 + miss):
                kf.correct(C[j, :2].astype(np.float32).reshape(2, 1)); acc[fi] = C[j, :2]; miss = 0; continue
        miss += 1
        if miss > BALL_MAX_MISS: kf, miss = None, 0
    if kf is None and len(C) and C[:, 2].max() >= BALL_INIT_CONF:
        j = int(C[:, 2].argmax()); kf = new_kf(*C[j, :2]); acc[fi] = C[j, :2]; reinit += 1
ball = pd.DataFrame(acc, columns=['bx', 'by'])
filled = ball.copy()
for sc in np.unique(scene_of):                            # riempimento dei buchi brevi, solo dentro la scena
    m = scene_of == sc
    filled.loc[m] = ball.loc[m].interpolate(limit=BALL_FILL, limit_area='inside')
ball = filled
valid_frames = (~excluded).sum()
print(f'palla: candidati in {len(cand_by) / valid_frames:.0%} dei frame | accettata in {np.isfinite(acc[:, 0]).sum() / valid_frames:.0%} '
      f'| dopo riempimento {ball.bx.notna().sum() / valid_frames:.0%} | riagganci: {reinit}')

## 6. Possesso ed eventi automatici (regole del notebook 14 v2, mai attraverso un taglio di scena)

In [ ]:
by_frame = dict(tuple(tracks[tracks.team >= 0].groupby('frame')))
raw = np.full(N, -1)
for fi in range(N):
    g = by_frame.get(fi)
    if g is None or excluded[fi]: continue
    p = g[(g.cls == 'player-in-possession') & (g.cls_conf >= POSS_CONF)]
    bx, by = ball.bx[fi], ball.by[fi]
    if len(p):
        if len(p) > 1 and not np.isnan(bx):
            p = p.assign(d=np.hypot((p.x1 + p.x2) / 2 - bx, (p.y1 + p.y2) / 2 - by)).sort_values('d')
        raw[fi] = int(p.track_id.iloc[0]); continue
    if np.isnan(bx): continue
    h = (g.y2 - g.y1).values
    dx = np.maximum.reduce([g.x1.values - bx, np.zeros(len(g)), bx - g.x2.values])
    dy = np.maximum.reduce([g.y1.values - by, np.zeros(len(g)), by - g.y2.values])
    d = np.hypot(dx, dy) / np.maximum(h, 1); k = int(d.argmin())
    if d[k] <= POSS_RADIUS: raw[fi] = int(g.track_id.values[k])
poss = np.full(N, -1); cur, cand, cnt, cand_start, lost = -1, -1, 0, 0, 0
for fi in range(N):
    if fi > 0 and scene_of[fi] != scene_of[fi - 1]: cur, cand, cnt, lost = -1, -1, 0, 0
    r = raw[fi]
    if r == -1:
        lost += 1
        if lost > POSS_LOST: cur = -1
    else:
        lost = 0
        if r == cur: cand, cnt = -1, 0
        else:
            if r == cand: cnt += 1
            else: cand, cnt, cand_start = r, 1, fi
            if cnt >= POSS_MIN: cur = cand; poss[cand_start:fi] = cur; cand, cnt = -1, 0
    poss[fi] = cur
segs, s = [], None
for fi in range(N + 1):
    v = poss[fi] if fi < N else -2
    if s is not None and (v != s[0] or (fi < N and scene_of[fi] != scene_of[s[1]])):
        segs.append({'track_id': s[0], 'start': s[1], 'end': fi - 1}); s = None
    if s is None and fi < N and v >= 0: s = (v, fi)
sg = pd.DataFrame(segs, columns=['track_id', 'start', 'end'])
sg['team'] = sg.track_id.map(team_of).fillna(-1).astype(int); sg = sg[sg.team >= 0]
n_raw = len(sg); changed = True
while changed and len(sg):
    changed = False
    short = sg[(sg.end - sg.start + 1) / FPS < MIN_SEG_S]
    if len(short): sg = sg.drop(index=short.index); changed = True
    sg = sg.sort_values('start').reset_index(drop=True); merged = []
    for r in sg.to_dict('records'):
        if merged and merged[-1]['track_id'] == r['track_id'] and (r['start'] - merged[-1]['end']) / FPS <= PASS_MAX_GAP_S:
            merged[-1]['end'] = r['end']; changed = True
        else: merged.append(dict(r))
    sg = pd.DataFrame(merged, columns=['track_id', 'start', 'end', 'team'])
if len(sg):
    sg['team_run'] = ((sg.team != sg.team.shift()) | (scene_of[sg.start.values] != scene_of[sg.start.shift().fillna(0).astype(int).values])).cumsum()
    run_dur = (sg.groupby('team_run').end.max() - sg.groupby('team_run').start.min() + 1) / FPS
    sg['team_run_s'] = sg.team_run.map(run_dur)
segs = sg
poss = np.full(N, -1)
for r in segs.itertuples(): poss[int(r.start):int(r.end) + 1] = int(r.track_id)
print(f'possesso: {n_raw} segmenti grezzi -> {len(segs)} dopo la pulizia')

def pos_at(tid, fi, win=15):
    g = tracks[(tracks.track_id == tid) & tracks.xs_ft.notna() & tracks.frame.between(fi - win, fi + win)]
    if not len(g): return (np.nan, np.nan)
    r = g.iloc[(g.frame - fi).abs().argmin()]; return float(r.xs_ft), float(r.ys_ft)
R3 = court.three_point_arc_radius; STRAIGHT = court.straight_section_three_point_line
CORNER = W_FT / 2 - court.sideline_to_three_point_line
BASKETS = [(court.baseline_to_rim_center, W_FT / 2), (L_FT - court.baseline_to_rim_center, W_FT / 2)]
def shot_value(x, y):
    if np.isnan(x): return np.nan, np.nan
    bx, byy = min(BASKETS, key=lambda b: abs(b[0] - x)); d = float(np.hypot(x - bx, y - byy))
    from_baseline = x if bx < L_FT / 2 else L_FT - x
    three = abs(y - byy) >= CORNER if from_baseline <= STRAIGHT else d >= R3
    return (3 if three else 2), d * FT_TO_M

sh = tracks[tracks.cls.isin(['player-jump-shot', 'player-layup-dunk']) & (tracks.cls_conf >= 0.3) & (tracks.team >= 0)]
cand_s = []
for tid, g in sh.groupby('track_id'):
    f = g.frame.values
    for r in np.split(f, np.where(np.diff(f) > 3)[0] + 1):
        if len(r) >= SHOT_MIN_FRAMES: cand_s.append((int(r[0]), int(tid), len(r)))
shots = []
for f0, tid, n in sorted(cand_s):
    if shots and f0 - shots[-1]['frame'] < 1.5 * FPS:
        if n > shots[-1]['_n']: shots[-1].update(frame=f0, track_id=tid, _n=n)
        continue
    shots.append({'frame': f0, 'track_id': tid, '_n': n})
made_f = np.sort(det[(det['name'] == 'ball-in-basket') & (det.conf >= MADE_CONF)].frame.unique())
events = []
for s in shots:
    w = made_f[(made_f >= s['frame']) & (made_f <= s['frame'] + SHOT_WINDOW_S * FPS) & (scene_of[made_f] == scene_of[s['frame']])]
    made = len(w) >= MADE_MIN_FRAMES
    x, y = pos_at(s['track_id'], s['frame']); val, dist = shot_value(x, y)
    events.append({'tipo': 'tiro_segnato' if made else 'tiro_sbagliato', 'frame': s['frame'], 'da': s['track_id'], 'a': np.nan,
                   'team': team_of.get(s['track_id'], -1), 'x_ft': x, 'y_ft': y, 'x2_ft': np.nan, 'y2_ft': np.nan,
                   'dettaglio': f'da {val} punti, {dist:.1f} m' if not np.isnan(dist) else ''})
shot_frames = np.array([s['frame'] for s in shots]); shot_made = {s['frame']: e['tipo'] == 'tiro_segnato' for s, e in zip(shots, events)}
for i in range(len(segs) - 1):
    a, b = segs.iloc[i], segs.iloc[i + 1]
    gap = (b.start - a.end) / FPS
    if gap > PASS_MAX_GAP_S or a.track_id == b.track_id or scene_of[int(a.end)] != scene_of[int(b.start)]: continue
    xa, ya = pos_at(a.track_id, a.end); xb, yb = pos_at(b.track_id, b.start)
    between = shot_frames[(shot_frames >= a.start) & (shot_frames <= b.start)] if len(shot_frames) else []
    base = {'frame': int(b.start), 'da': int(a.track_id), 'a': int(b.track_id), 'team': int(b.team), 'x_ft': xa, 'y_ft': ya, 'x2_ft': xb, 'y2_ft': yb}
    if len(between):
        if not shot_made[int(between[-1])]:
            events.append({**base, 'tipo': 'rimbalzo', 'dettaglio': 'offensivo' if a.team == b.team else 'difensivo'})
        continue
    if a.team == b.team:
        d = np.hypot(xa - xb, ya - yb) * FT_TO_M
        if not np.isnan(d) and d < max(PASS_MIN_M, SWITCH_DIST_M): continue
        events.append({**base, 'frame': int(a.end), 'tipo': 'passaggio', 'dettaglio': f'{d:.1f} m' if not np.isnan(d) else ''})
    elif a.team_run_s >= STEAL_MIN_S and b.team_run_s >= STEAL_MIN_S:
        events.append({**base, 'tipo': 'palla_rubata', 'dettaglio': f'squadra {int(a.team)} -> {int(b.team)}'})
COLS = ['tipo', 'frame', 'da', 'a', 'team', 'x_ft', 'y_ft', 'x2_ft', 'y2_ft', 'dettaglio']
ev_auto = pd.DataFrame(events, columns=COLS).sort_values('frame').reset_index(drop=True)
ev_auto.insert(2, 'tempo_s', (ev_auto.frame / FPS + START_S).round(2))
ev_auto.to_csv(OUT / 'eventi_automatici.csv', index=False)
print('eventi automatici:', ev_auto.tipo.value_counts().to_dict())
ev_auto.round(1)

## 7. Eventi manuali (lavagna semi-automatica)

Guarda il **video diagnostico** della sezione 9 (gli ID dei giocatori sono scritti sopra le box) e scrivi in `annotations/eventi_manuali.csv` una riga per evento:

```
clip,tipo,tempo_s,da,a,note
nba_clip_01,passaggio,12.4,1007,1003,
nba_clip_01,tiro_sbagliato,18.0,1003,,da 3
nba_clip_01,rimbalzo,19.1,2004,,difensivo
nba_clip_01,palla_rubata,25.6,2004,1009,
nba_clip_01,tiro_segnato,31.2,1009,,
```
`tempo_s` è il tempo **del video originale** (come lo vedi nel lettore); `da` = chi passa / tira / perde palla, `a` = chi riceve / ruba. Se non sai l'ID lascia vuoto: l'evento entra comunque nella valutazione, ma non nella lavagna. Le posizioni sul campo vengono prese in automatico dalle tracce.

In [ ]:
if not MANUAL.is_file():
    pd.DataFrame(columns=['clip', 'tipo', 'tempo_s', 'da', 'a', 'note']).to_csv(MANUAL, index=False)
    print('Creato il modello vuoto:', MANUAL)
man = pd.read_csv(MANUAL); man = man[man['clip'] == CLIP_ID].copy()
rows = []
for r in man.itertuples():
    fi = int(round((r.tempo_s - START_S) * FPS))
    if not 0 <= fi < N: continue
    da = int(r.da) if pd.notna(r.da) else None; a = int(r.a) if pd.notna(r.a) else None
    x, y = pos_at(da, fi) if da else (np.nan, np.nan)
    x2, y2 = pos_at(a, fi + int(.5 * FPS)) if a else (np.nan, np.nan)
    ref_tid = a if (r.tipo == 'palla_rubata' and a) else (da or a)
    team = team_of.get(ref_tid, -1) if ref_tid else -1
    det_txt = ''
    if str(r.tipo).startswith('tiro'):
        val, dist = shot_value(x, y); det_txt = f'da {val} punti, {dist:.1f} m' if not np.isnan(dist) else ''
    rows.append({'tipo': r.tipo, 'frame': fi, 'da': da, 'a': a, 'team': team, 'x_ft': x, 'y_ft': y, 'x2_ft': x2, 'y2_ft': y2,
                 'dettaglio': det_txt or (r.note if isinstance(r.note, str) else '')})
ev_man = pd.DataFrame(rows, columns=COLS).sort_values('frame').reset_index(drop=True)
ev_man.insert(2, 'tempo_s', (ev_man.frame / FPS + START_S).round(2))
print(f'eventi manuali per {CLIP_ID}: {len(ev_man)}', ev_man.tipo.value_counts().to_dict() if len(ev_man) else '')

## 8. Lavagna tattica e linea del tempo (automatica e, se presente, manuale)

In [ ]:
SCALE, PAD = 10, 30
base = draw_court(config=court, scale=SCALE, padding=PAD)
def to_map(x, y): return int(x * SCALE) + PAD, int(y * SCALE) + PAD
TEAM_RGB = {0: '#1e8cff', 1: '#e62828', -1: '#999999'}
from matplotlib.lines import Line2D

def lavagna(ev, title, path):
    fig, ax = plt.subplots(figsize=(13, 7.5)); ax.imshow(cv2.cvtColor(base, cv2.COLOR_BGR2RGB))
    for r in ev.itertuples():
        if np.isnan(r.x_ft) and r.tipo != 'palla_rubata': continue
        c = TEAM_RGB.get(int(r.team), '#999')
        if r.tipo == 'passaggio' and not np.isnan(r.x2_ft):
            ax.annotate('', xy=to_map(r.x2_ft, r.y2_ft), xytext=to_map(r.x_ft, r.y_ft), arrowprops=dict(arrowstyle='->', color=c, lw=2, alpha=.85))
        elif r.tipo == 'palla_rubata':
            xx, yy = (r.x2_ft, r.y2_ft) if not np.isnan(r.x2_ft) else (r.x_ft, r.y_ft)
            if not np.isnan(xx): ax.scatter(*to_map(xx, yy), marker='*', s=420, color='gold', edgecolor=c, linewidth=2, zorder=5)
        elif r.tipo == 'tiro_segnato': ax.scatter(*to_map(r.x_ft, r.y_ft), s=260, color='#2ca02c', edgecolor=c, linewidth=3, zorder=6)
        elif r.tipo == 'tiro_sbagliato': ax.scatter(*to_map(r.x_ft, r.y_ft), s=260, marker='X', color='#d62728', edgecolor=c, linewidth=1.5, zorder=6)
        elif r.tipo == 'rimbalzo': ax.scatter(*to_map(r.x_ft, r.y_ft), s=120, marker='s', color='white', edgecolor=c, linewidth=2, zorder=4)
    leg = [Line2D([], [], color=TEAM_RGB[0], lw=2, label='passaggio squadra 0'), Line2D([], [], color=TEAM_RGB[1], lw=2, label='passaggio squadra 1'),
           Line2D([], [], marker='o', ls='', color='#2ca02c', ms=12, label='tiro segnato'), Line2D([], [], marker='X', ls='', color='#d62728', ms=12, label='tiro sbagliato'),
           Line2D([], [], marker='*', ls='', color='gold', ms=16, label='palla rubata'), Line2D([], [], marker='s', ls='', mfc='white', mec='k', ms=9, label='rimbalzo')]
    ax.legend(handles=leg, loc='upper center', bbox_to_anchor=(.5, -.02), ncol=3, frameon=False)
    ax.set_title(title); ax.axis('off'); fig.tight_layout(); fig.savefig(path, dpi=170); plt.show()

lavagna(ev_auto, f'{CLIP_ID} — eventi automatici', OUT / 'lavagna_automatica.png')
if len(ev_man): lavagna(ev_man, f'{CLIP_ID} — eventi annotati (posizioni automatiche)', OUT / 'lavagna_manuale.png')

fig, ax = plt.subplots(figsize=(14, 3))
for r in segs.itertuples():
    ax.barh(0, (r.end - r.start + 1) / FPS, left=r.start / FPS + START_S, color=TEAM_RGB.get(int(r.team), '#999'), height=.5)
for c in cuts: ax.axvline(c / FPS + START_S, color='k', ls=':', lw=1)
for a, b in ESCLUDI: ax.axvspan(a, b, color='#ccc', alpha=.6)
mk = {'passaggio': ('^', 'k'), 'palla_rubata': ('*', 'gold'), 'tiro_segnato': ('o', '#2ca02c'), 'tiro_sbagliato': ('X', '#d62728'), 'rimbalzo': ('s', 'w')}
for y0, ev, lab in [(.55, ev_auto, 'auto'), (.85, ev_man, 'manuale')]:
    for t, (m, c) in mk.items():
        e = ev[ev.tipo == t] if len(ev) else ev
        if len(e): ax.scatter(e.tempo_s, np.full(len(e), y0), marker=m, color=c, edgecolor='k', s=80, zorder=5, label=t if lab == 'auto' else None)
ax.set_yticks([.55, .85]); ax.set_yticklabels(['automatici', 'manuali']); ax.set_ylim(-.4, 1.1)
ax.set_xlabel('tempo nel video (s)'); ax.set_xlim(START_S, START_S + N / FPS)
ax.set_title('Possesso automatico (blu = squadra 0, rosso = squadra 1), tagli di scena (:) ed eventi')
ax.legend(ncol=5, loc='upper center', bbox_to_anchor=(.5, -.3), frameon=False); fig.tight_layout(); fig.savefig(OUT / 'linea_del_tempo.png', dpi=160); plt.show()

## 9. Video diagnostico: ID, squadre, palla (grigio = candidati, giallo = accettata), possesso, eventi e lavagna

In [ ]:
EV_VIDEO = ev_man if len(ev_man) else ev_auto          # nel video: eventi manuali se ci sono, altrimenti automatici
mh = H_IMG; mw = int(base.shape[1] * mh / base.shape[0]); out_w = W_IMG + mw; out_w += out_w % 2
vw = cv2.VideoWriter(str(OUT / 'diagnostica_raw.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), FPS, (out_w, H_IMG))
trk_by = dict(tuple(tracks.groupby('frame')))
LABEL = {'passaggio': 'PASSAGGIO', 'palla_rubata': 'PALLA RUBATA', 'tiro_segnato': 'TIRO SEGNATO', 'tiro_sbagliato': 'TIRO SBAGLIATO', 'rimbalzo': 'RIMBALZO'}
cut_set = set(cuts)
for fi, f in frames_iter(desc='video'):
    img = f.copy(); mp = base.copy()
    if excluded[fi]:
        cv2.putText(img, 'ESCLUSO', (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
    for r in EV_VIDEO[(EV_VIDEO.frame <= fi) & EV_VIDEO.tipo.str.startswith('tiro')].itertuples():
        if np.isnan(r.x_ft): continue
        p = to_map(r.x_ft, r.y_ft)
        if r.tipo == 'tiro_segnato': cv2.circle(mp, p, 10, (44, 160, 44), 3)
        else: cv2.drawMarker(mp, p, (40, 39, 214), cv2.MARKER_TILTED_CROSS, 20, 3)
    for r in EV_VIDEO[(EV_VIDEO.frame <= fi) & (EV_VIDEO.frame > fi - 2 * FPS) & EV_VIDEO.tipo.isin(['passaggio', 'palla_rubata'])].itertuples():
        if np.isnan(r.x_ft) or np.isnan(r.x2_ft): continue
        cv2.arrowedLine(mp, to_map(r.x_ft, r.y_ft), to_map(r.x2_ft, r.y2_ft), TEAM_COLORS.get(int(r.team), (0, 215, 255)), 3, tipLength=.08)
    holder = poss[fi]
    for r in trk_by.get(fi, pd.DataFrame()).itertuples():
        c = TEAM_COLORS.get(int(r.team), (160, 160, 160)); th = 4 if r.track_id == holder else 2
        cv2.rectangle(img, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), c, th)
        cv2.putText(img, str(r.track_id), (int(r.x1), max(14, int(r.y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, .55, c, 2)
        if not np.isnan(r.xs_ft):
            p = to_map(r.xs_ft, r.ys_ft); cv2.circle(mp, p, 9, c, -1); cv2.circle(mp, p, 9, (0, 0, 0), 1)
            if r.track_id == holder: cv2.circle(mp, p, 14, (0, 200, 255), 3)
    for cx, cy, cf in cand_by.get(fi, []):
        cv2.circle(img, (int(cx), int(cy)), 6, (150, 150, 150), 1)
    if not np.isnan(ball.bx[fi]):
        cv2.circle(img, (int(ball.bx[fi]), int(ball.by[fi])), 9, (0, 220, 255), 3)
    for k, r in enumerate(EV_VIDEO[(EV_VIDEO.frame <= fi) & (EV_VIDEO.frame > fi - 1.5 * FPS)].itertuples()):
        txt = LABEL.get(r.tipo, r.tipo) + (f'  {r.da}->{r.a}' if r.tipo in ('passaggio', 'palla_rubata') else f'  {r.da}  {r.dettaglio}')
        cv2.putText(img, txt, (12, 34 + 28 * k), cv2.FONT_HERSHEY_SIMPLEX, .75, (0, 0, 0), 5)
        cv2.putText(img, txt, (12, 34 + 28 * k), cv2.FONT_HERSHEY_SIMPLEX, .75, (255, 255, 255), 2)
    t = f'{fi / FPS + START_S:6.1f} s'
    cv2.putText(img, t, (W_IMG - 170, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 6)
    cv2.putText(img, t, (W_IMG - 170, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    if fi in cut_set: cv2.rectangle(img, (0, 0), (W_IMG - 1, H_IMG - 1), (0, 0, 255), 12)
    if Hs[fi] is None: cv2.putText(mp, 'campo non visibile', (10, 22), cv2.FONT_HERSHEY_SIMPLEX, .6, (0, 0, 255), 2)
    canvas = np.zeros((H_IMG, out_w, 3), np.uint8); canvas[:, :W_IMG] = img; canvas[:, W_IMG:W_IMG + mw] = cv2.resize(mp, (mw, mh))
    vw.write(canvas)
vw.release()
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(OUT / 'diagnostica_raw.mp4'), '-c:v', 'libx264', '-crf', '24',
                '-pix_fmt', 'yuv420p', str(OUT / 'diagnostica.mp4')], check=True)
(OUT / 'diagnostica_raw.mp4').unlink()
print('Video:', OUT / 'diagnostica.mp4')

## 10. Valutazione degli eventi automatici contro quelli annotati

In [ ]:
def match(pred_t, ref_t, tol):
    pairs = sorted((abs(p - r), i, j) for i, p in enumerate(pred_t) for j, r in enumerate(ref_t) if abs(p - r) <= tol)
    up, ur = set(), set()
    for _, i, j in pairs:
        if i not in up and j not in ur: up.add(i); ur.add(j)
    return len(up)
if len(man):
    groups = {t: [t] for t in ['passaggio', 'palla_rubata', 'tiro_segnato', 'tiro_sbagliato', 'rimbalzo']}
    groups['tiro (tentativo)'] = ['tiro_segnato', 'tiro_sbagliato']
    rows = []
    for name, types in groups.items():
        p = ev_auto[ev_auto.tipo.isin(types)].tempo_s.values; r = man[man.tipo.isin(types)].tempo_s.values
        tp = match(p, r, EVENT_TOL_S)
        rows.append({'tipo': name, 'rilevati': len(p), 'veri': len(r), 'corretti': tp,
                     'precisione': tp / len(p) if len(p) else np.nan, 'richiamo': tp / len(r) if len(r) else np.nan})
    val = pd.DataFrame(rows).round(2); val.insert(0, 'clip', CLIP_ID)
    val.to_csv(OUT / 'valutazione_eventi.csv', index=False); display(val)
else:
    print('Nessun evento annotato per questa clip: compila', MANUAL.name, 'e riesegui le sezioni 7-10.')

summary = {'clip': CLIP_ID, 'secondi': round(N / FPS, 1), 'tagli_scena': len(cuts),
           'palla_candidati_%': round(100 * len(cand_by) / max(1, (~excluded).sum()), 1),
           'palla_accettata_%': round(100 * np.isfinite(acc[:, 0]).sum() / max(1, (~excluded).sum()), 1),
           'tracce': int(tracks.track_id.nunique()), 'segmenti_possesso': len(segs),
           'eventi_automatici': ev_auto.tipo.value_counts().to_dict(), 'eventi_manuali': len(ev_man)}
(OUT / 'riepilogo.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False)); print(json.dumps(summary, indent=2, ensure_ascii=False))